# Week 7 Assignment - Document Question Answering System (RAG)

Data Science Internship - Dhanraj Deshmukh

This notebook builds a Retrieval-Augmented Generation pipeline that
answers questions grounded in a custom document instead of relying on a
language model's own internal knowledge. The three pieces the assignment
asks for map onto three stages here:

- **Retrieval** - finding the chunks of the document that are actually
  relevant to the question, using embeddings and vector similarity
- **Augmentation** - adding those retrieved chunks into the model's input
  so it has real context to work from
- **Generation** - a language model producing the final answer using
  that retrieved context, so the answer stays grounded in the source
  document rather than the model just guessing

The document ingestion module below accepts three input types: a real
PDF you upload, a plain text file, or a document pulled from a Hugging
Face dataset. A short demo document (internship program guidelines) is
used as the default fallback so the notebook still runs end to end if
nothing is uploaded, but the upload path is fully wired up in Section 2.

## 1. Setup

Installing what's needed for the pipeline:
- `pypdf` for reading PDFs
- `langchain-text-splitters` for chunking
- `sentence-transformers` for both the embedding model and the
  cross-encoder used later for re-ranking
- `chromadb` as the vector store
- `transformers` for the generation model
- `datasets` for pulling a document from Hugging Face instead of a local
  file, if that's the input type being used

In [1]:
!pip install -q pypdf langchain-text-splitters sentence-transformers chromadb transformers accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

In [2]:
import os
import time
import textwrap

import numpy as np
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from transformers import pipeline
from datasets import load_dataset

from pypdf import PdfReader

print("all imports loaded fine")

all imports loaded fine


## 2. Document Ingestion

`load_document()` covers the three input types the assignment asks for:
a real PDF, a plain text file, or a Hugging Face dataset. Each returns
the same thing - a single raw text string - so everything downstream
doesn't need to know or care which source it came from.

In [3]:
def load_document(source, source_type="auto", hf_config=None):
    """
    Loads raw text from:
      - a PDF file path                 (source_type='pdf')
      - a plain text file path          (source_type='txt')
      - a raw string already in memory  (source_type='raw')
      - a Hugging Face dataset          (source_type='huggingface')

    For 'huggingface', `source` is the dataset name (e.g.
    'vectara/open_ragbench') and `hf_config` can optionally specify
    {'split': ..., 'text_field': ..., 'num_docs': ...} to control which
    rows get pulled and which column holds the text.
    """
    if source_type == "raw":
        return source

    if source_type == "huggingface":
        cfg = hf_config or {}
        split = cfg.get("split", "train")
        text_field = cfg.get("text_field", "text")
        num_docs = cfg.get("num_docs", 20)

        dataset = load_dataset(source, split=f"{split}[:{num_docs}]")
        documents = [row[text_field] for row in dataset if row.get(text_field)]
        return "\n\n".join(documents)

    if source_type == "auto":
        if source.lower().endswith(".pdf"):
            source_type = "pdf"
        elif source.lower().endswith(".txt"):
            source_type = "txt"
        else:
            raise ValueError("Could not infer file type, pass source_type explicitly")

    if source_type == "pdf":
        reader = PdfReader(source)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        return text

    if source_type == "txt":
        with open(source, "r", encoding="utf-8") as f:
            return f.read()

    raise ValueError(f"Unsupported source_type: {source_type}")

### 2.1 Upload your own document (PDF or TXT)

Running this cell pops up a real file picker in Colab. Upload a PDF or a
TXT file - a resume, notes, a research paper, whatever - and it gets
loaded straight into `document_text`, which is what the rest of the
notebook runs on. If this is run outside Colab, or the cell is skipped,
it falls back to the demo document defined in 2.2 so the notebook still
works without any manual step.

In [4]:
document_text = None
uploaded_filename = None

try:
    from google.colab import files
    print("Upload a PDF or TXT file (or just don't select anything to skip and use the demo document):")
    uploaded = files.upload()

    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        document_text = load_document(uploaded_filename)
        print(f"\nloaded '{uploaded_filename}', {len(document_text)} characters")
    else:
        print("\nno file uploaded, will fall back to the demo document below")

except ImportError:
    print("not running in Colab, skipping the upload widget - using the demo document below")

Upload a PDF or TXT file (or just don't select anything to skip and use the demo document):


Saving Celebal_Internship_Progress_Report.pdf to Celebal_Internship_Progress_Report.pdf

loaded 'Celebal_Internship_Progress_Report.pdf', 6992 characters


### 2.2 Demo document (fallback)

Used automatically if nothing was uploaded above. This keeps the
notebook fully runnable end to end without depending on a file being
present in the session.

In [5]:
demo_document = """
Celebal Technologies Data Science Internship Program

The internship runs for a fixed duration and is organized into weekly
assignments, each covering a different part of the data science and
machine learning workflow. Interns are expected to submit a completed
notebook for every weekly assignment, along with a short README
explaining the approach taken and the key results.

Weeks 1 through 3 focus on foundational statistics, exploratory data
analysis, and unsupervised learning techniques such as K-Means and
DBSCAN clustering. Interns work with real-world tabular datasets and are
expected to justify every preprocessing decision, not just apply
transformations blindly.

Weeks 4 through 6 shift toward deep learning. This includes image
classification using both artificial neural networks and convolutional
neural networks, sequence modeling for text generation using RNN, LSTM,
and GRU architectures, and unsupervised representation learning using
autoencoders, including denoising autoencoders trained on noisy image
data.

Week 7 introduces Retrieval-Augmented Generation, where interns build a
pipeline that can answer questions grounded in a custom document rather
than relying purely on a language model's internal knowledge. This
involves chunking text, generating embeddings, storing them in a vector
database, and retrieving relevant context at query time.

Evaluation for each week is based on correctness of implementation,
clarity of the analysis written in markdown cells, and code quality.
Interns are encouraged to go beyond the minimum requirements by trying
additional experiments, such as comparing architectures, tuning
hyperparameters, or testing alternative retrieval strategies, and to
document what they observed rather than just reporting a final number.

Mentorship sessions are held weekly, where interns can ask questions
about blockers from the previous assignment and get guidance on the
upcoming one. Interns are expected to come prepared with specific
questions rather than general status updates.
"""

if document_text is None:
    document_text = demo_document
    print("using the demo document")
else:
    print(f"using the uploaded document ('{uploaded_filename}')")

print(f"\nfinal document length: {len(document_text)} characters")

using the uploaded document ('Celebal_Internship_Progress_Report.pdf')

final document length: 6992 characters


### 2.3 Optional: load from a Hugging Face dataset instead

The assignment resources also point to a Hugging Face dataset
(`vectara/open_ragbench`) as an alternative source. This cell is not run
by default since the uploaded/demo document above is already set as
`document_text`, but it shows the same ingestion function handling a
completely different source type. Uncomment to actually pull it in.

In [6]:
# hf_text = load_document(
#     "vectara/open_ragbench",
#     source_type="huggingface",
#     hf_config={"split": "train", "text_field": "text", "num_docs": 20}
# )
# document_text = hf_text
# print(f"loaded {len(document_text)} characters from Hugging Face dataset")

## 3. Text Chunking

Splitting the document into overlapping chunks so retrieval has smaller,
more focused pieces of text to search over instead of matching against
the whole document at once. `chunk_overlap` keeps some continuity between
chunks so a sentence cut at a boundary doesn't lose all its context.

In [7]:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(document_text)

print(f"document split into {len(chunks)} chunks")
print("\nfirst chunk:\n")
print(chunks[0])

document split into 29 chunks

first chunk:

Data Science Internship – Progress Report
Celebal Technologies
Intern:
Dhanraj Deshmukh
Program:
Data Science Internship (Celebal Technologies Externship Internship – CEI 2026)
Reporting Period:
Week 1 – Week 4 (Week 5 currently in progress)
Submitted To:
Internship Mentor
Date:
July 15, 2026


## 4. Embedding Model

Using `all-MiniLM-L6-v2` from sentence-transformers - small enough to run
comfortably on CPU, gives 384-dimensional embeddings, and is a common
default for RAG prototypes so it's a reasonable starting point before
trying anything heavier.

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print("embedding shape:", chunk_embeddings.shape)
print("embedding dimension:", chunk_embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (29, 384)
embedding dimension: 384


## 5. Vector Database

Storing the chunk embeddings in a ChromaDB collection. Chroma handles the
similarity search under the hood, so retrieval later on is a single
query call instead of writing a nearest-neighbor search by hand.

In [9]:
chroma_client = chromadb.Client()

# fresh collection every run so re-running the notebook doesn't duplicate entries
try:
    chroma_client.delete_collection("week7_documents")
except Exception:
    pass

collection = chroma_client.create_collection(name="week7_documents")

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=[{"chunk_index": i} for i in range(len(chunks))]
)

print("stored", collection.count(), "chunks in the vector store")

stored 29 chunks in the vector store


## 6. Query Embedding and Retrieval

The question gets embedded with the exact same model used for the
document chunks - mixing embedding models between the document side and
the query side would make the similarity scores meaningless. Chroma's
`query()` handles the actual nearest-neighbor lookup and returns both the
matching chunks and their distance scores.

In [10]:
def retrieve_context(question, top_k=3):
    query_embedding = embedding_model.encode([question])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    retrieved_chunks = results["documents"][0]
    distances = results["distances"][0]
    return retrieved_chunks, distances

In [11]:
test_chunks, test_distances = retrieve_context("What happens in week 7 of the internship?")
for chunk, dist in zip(test_chunks, test_distances):
    print(f"distance: {dist:.4f}")
    print(textwrap.fill(chunk, 90))
    print("-" * 60)

distance: 1.0640
Internship Mentor Date: July 15, 2026 Repository: github.com/Johanl001/Celebal_Tech This
report summarizes technical work completed across the first four weeks of the internship,
covering statistical/mathematical foundations, an end-to-end regression & time-series ML
pipeline, a hybrid
------------------------------------------------------------
distance: 1.1847
flagged as an area to revisit (e.g., relaxing the patience parameter or reducing
augmentation strength). Week 5 — In Progress Currently working through Week 5 (Model
Evaluation & Validation). Detailed findings will be included in the next progress update.
Skills Gained — Summary 
------------------------------------------------------------
distance: 1.1871
Data Science Internship – Progress Report Celebal Technologies Intern: Dhanraj Deshmukh
Program: Data Science Internship (Celebal Technologies Externship Internship – CEI 2026)
Reporting Period: Week 1 – Week 4 (Week 5 currently in progress) Submitted To: In

## 7. Answer Generation

Retrieved chunks get stitched into a prompt along with the question, and
a language model generates the final answer from that context. Using
`flan-t5-base` here since it runs locally without needing an API key,
which keeps the notebook runnable end to end on a free Colab instance.
Swapping in a larger hosted model (GPT-4, Claude, Llama) would give
noticeably better answers - this is just the version that doesn't depend
on anyone's API key to run.

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# calling model.generate() directly instead of pipeline("text2text-generation", ...) -
# some transformers builds (this Colab environment included) no longer register
# that task alias, so going straight through the model/tokenizer avoids the
# pipeline task lookup entirely and works regardless of which aliases are registered
model_name = "google/flan-t5-base"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
qa_model = qa_model.to(device)

def generate_answer(prompt, max_new_tokens=100):
    inputs = qa_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = qa_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,            # beam search instead of greedy decoding - fewer rambling/repeated answers
        no_repeat_ngram_size=3, # blocks the model from repeating the same 3-word phrase
        early_stopping=True,
    )
    return qa_tokenizer.decode(output_ids[0], skip_special_tokens=True)

def build_prompt(question, context_chunks):
    context = "\n\n".join(context_chunks)
    return f"""Answer the question in one or two sentences, using only the context below. If the answer is not in the context, say "I don't know based on the given context."

Context:
{context}

Question: {question}

Answer:"""

def answer_question(question, top_k=3, verbose=False):
    retrieved_chunks, distances = retrieve_context(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)

    if verbose:
        print("retrieved chunks (closest first):")
        for chunk, dist in zip(retrieved_chunks, distances):
            print(f"  distance {dist:.4f}: {chunk[:80]}...")

    response = generate_answer(prompt)
    return {
        "question": question,
        "answer": response,
        "retrieved_chunks": retrieved_chunks,
        "distances": distances
    }

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 8. Validation - Sample Questions

Running the full pipeline on a handful of different questions to check
that retrieval is actually pulling relevant chunks and the generated
answer stays grounded in them, rather than drifting off into something
the base model happens to know already. If a custom document was
uploaded in Section 2.1, swap these questions out for ones relevant to
that document.

In [13]:
sample_questions = [
    "What topics are covered in weeks 4 through 6 of the internship?",
    "How is Week 7 different from the earlier weeks?",
    "What is evaluation based on for each weekly assignment?",
    "How often are mentorship sessions held?",
]

validation_log = []

for question in sample_questions:
    start = time.time()
    result = answer_question(question, top_k=3, verbose=True)
    elapsed = time.time() - start

    print(f"\nQ: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"(retrieved in {elapsed:.2f}s)")
    print("=" * 70)

    validation_log.append({
        "question": question,
        "answer": result["answer"],
        "top_chunk_distance": min(result["distances"]),
        "chunks_retrieved": len(result["retrieved_chunks"]),
        "response_time_s": round(elapsed, 2)
    })

validation_df = pd.DataFrame(validation_log)
validation_df

retrieved chunks (closest first):
  distance 1.0191: Internship Mentor
Date:
July 15, 2026
Repository:
github.com/Johanl001/Celebal_T...
  distance 1.1083: Data Science Internship – Progress Report
Celebal Technologies
Intern:
Dhanraj D...
  distance 1.1570: flagged as an area to revisit (e.g., relaxing the patience parameter or reducing...

Q: What topics are covered in weeks 4 through 6 of the internship?
A: statistical/mathematical foundations, an end-to-end regression & time-series ML pipeline
(retrieved in 7.96s)
retrieved chunks (closest first):
  distance 1.3565: flagged as an area to revisit (e.g., relaxing the patience parameter or reducing...
  distance 1.5758: 17,560, RMSE = 18,725, R² = −2.22 — indicating the model struggled to generalize...
  distance 1.5964: Week 2 — End-to-End ML Pipeline: Tesla EV Sales
Status: Completed  |  Dataset: T...

Q: How is Week 7 different from the earlier weeks?
A: I don't know based on the given context.
(retrieved in 3.16s)
retrieved chunks

,question,answer,top_chunk_distance,chunks_retrieved,response_time_s
0,What topics are covered in weeks 4 through 6 o...,"statistical/mathematical foundations, an end-t...",1.019085,3,7.96
1,How is Week 7 different from the earlier weeks?,I don't know based on the given context.,1.356543,3,3.16
2,What is evaluation based on for each weekly as...,Model evaluation and validation,1.035456,3,2.21
3,How often are mentorship sessions held?,I don't know,1.203882,3,2.32


## 9. System Metrics Report

Summarizing the configuration used in this run, since the assignment
specifically asks for a report covering chunking setup, embedding
dimensions, vector store choice, and the language model used.

In [14]:
system_report = {
    "Document source": "uploaded file" if uploaded_filename else "demo document",
    "Document length (chars)": len(document_text),
    "Chunk size": CHUNK_SIZE,
    "Chunk overlap": CHUNK_OVERLAP,
    "Total chunks stored": len(chunks),
    "Embedding model": "all-MiniLM-L6-v2",
    "Embedding dimension": chunk_embeddings.shape[1],
    "Vector store": "ChromaDB (in-memory client)",
    "Similarity metric": "Chroma default (cosine on normalized embeddings)",
    "Generation model": "google/flan-t5-base",
    "Retrieval top_k": 3,
}

pd.DataFrame(system_report.items(), columns=["Setting", "Value"])

,Setting,Value
0,Document source,uploaded file
1,Document length (chars),6992
2,Chunk size,300
3,Chunk overlap,50
4,Total chunks stored,29
5,Embedding model,all-MiniLM-L6-v2
6,Embedding dimension,384
7,Vector store,ChromaDB (in-memory client)
8,Similarity metric,Chroma default (cosine on normalized embeddings)
9,Generation model,google/flan-t5-base


## 10. Optimization Experiments

The assignment lists a few directions to try beyond the basic pipeline -
different chunk sizes, hybrid search, and a re-ranking layer. Trying a
version of each below.

### 10.1 Chunk size comparison

Rebuilding the vector store twice more with a smaller and a larger chunk
size, then comparing how many chunks each produces and how the closest
retrieval distance changes for the same test question. Smaller chunks
give more precise matches but lose surrounding context; larger chunks
keep more context but retrieval gets less targeted.

In [15]:
def build_store_with_chunk_size(chunk_size, overlap, collection_name):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    local_chunks = splitter.split_text(document_text)
    local_embeddings = embedding_model.encode(local_chunks)

    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    local_collection = chroma_client.create_collection(name=collection_name)
    local_collection.add(
        ids=[f"c_{i}" for i in range(len(local_chunks))],
        embeddings=local_embeddings.tolist(),
        documents=local_chunks
    )
    return local_collection, len(local_chunks)

test_question = "What happens in week 7 of the internship?"
test_query_embedding = embedding_model.encode([test_question]).tolist()

chunk_size_results = []
for size, overlap in [(150, 20), (300, 50), (600, 100)]:
    coll, n_chunks = build_store_with_chunk_size(size, overlap, f"week7_test_{size}")
    result = coll.query(query_embeddings=test_query_embedding, n_results=1)
    chunk_size_results.append({
        "chunk_size": size,
        "overlap": overlap,
        "num_chunks": n_chunks,
        "top_match_distance": result["distances"][0][0]
    })

pd.DataFrame(chunk_size_results)

,chunk_size,overlap,num_chunks,top_match_distance
0,150,20,67,0.987574
1,300,50,29,1.063955
2,600,100,14,1.172697


### 10.2 Hybrid search (keyword + vector)

Pure vector search can miss chunks that share exact keywords with the
query but phrase things differently in embedding space. Here the vector
similarity score is combined with a simple keyword overlap score, so
chunks that match on both dimensions rank higher than chunks that only
match on one.

In [16]:
def keyword_overlap_score(question, chunk):
    question_words = set(question.lower().split())
    chunk_words = set(chunk.lower().split())
    if not question_words:
        return 0.0
    return len(question_words & chunk_words) / len(question_words)

def hybrid_retrieve(question, top_k=3, vector_weight=0.7):
    query_embedding = embedding_model.encode([question])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=len(chunks))

    candidates = results["documents"][0]
    distances = results["distances"][0]

    scored = []
    for chunk, dist in zip(candidates, distances):
        vector_score = 1 / (1 + dist)  # convert distance to a similarity-like score
        keyword_score = keyword_overlap_score(question, chunk)
        combined = vector_weight * vector_score + (1 - vector_weight) * keyword_score
        scored.append((chunk, combined))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

hybrid_results = hybrid_retrieve("mentorship sessions weekly")
for chunk, score in hybrid_results:
    print(f"combined score: {score:.3f}")
    print(textwrap.fill(chunk, 90))
    print("-" * 60)

combined score: 0.371
 Deep learning: ANN vs CNN architecture design, regularization techniques (BatchNorm,
Dropout, EarlyStopping, data augmentation) in TensorFlow/Keras.  Engineering practice:
reproducible Jupyter notebooks, Git/GitHub version control with structured weekly
------------------------------------------------------------
combined score: 0.349
Internship Mentor Date: July 15, 2026 Repository: github.com/Johanl001/Celebal_Tech This
report summarizes technical work completed across the first four weeks of the internship,
covering statistical/mathematical foundations, an end-to-end regression & time-series ML
pipeline, a hybrid
------------------------------------------------------------
combined score: 0.302
flagged as an area to revisit (e.g., relaxing the patience parameter or reducing
augmentation strength). Week 5 — In Progress Currently working through Week 5 (Model
Evaluation & Validation). Detailed findings will be included in the next progress update.
Skills Gaine

### 10.3 Re-ranking with a cross-encoder model

This is a real re-ranking model, not a hand-written heuristic:
`cross-encoder/ms-marco-MiniLM-L-6-v2` from sentence-transformers scores
a (question, chunk) pair directly rather than comparing two separate
embeddings, which tends to be more accurate than plain vector similarity
- at the cost of being slower, since every candidate has to be scored
individually. That's exactly why it's used as a second pass over a
smaller candidate pool instead of as the primary retrieval step: pull the
top 10 by fast vector search, then let the cross-encoder re-sort just
those down to the final top 3.

In [17]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_with_rerank(question, initial_k=10, final_k=3):
    query_embedding = embedding_model.encode([question])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=initial_k)
    candidates = results["documents"][0]

    pairs = [(question, chunk) for chunk in candidates]
    rerank_scores = cross_encoder.predict(pairs)

    reranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)
    return reranked[:final_k]

reranked_results = retrieve_with_rerank("evaluation criteria for weekly assignments")
for chunk, score in reranked_results:
    print(f"cross-encoder score: {score:.3f}")
    print(textwrap.fill(chunk, 90))
    print("-" * 60)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

cross-encoder score: -5.758
flagged as an area to revisit (e.g., relaxing the patience parameter or reducing
augmentation strength). Week 5 — In Progress Currently working through Week 5 (Model
Evaluation & Validation). Detailed findings will be included in the next progress update.
Skills Gained — Summary 
------------------------------------------------------------
cross-encoder score: -10.519
 Deep learning: ANN vs CNN architecture design, regularization techniques (BatchNorm,
Dropout, EarlyStopping, data augmentation) in TensorFlow/Keras.  Engineering practice:
reproducible Jupyter notebooks, Git/GitHub version control with structured weekly
------------------------------------------------------------
cross-encoder score: -10.844
17,560, RMSE = 18,725, R² = −2.22 — indicating the model struggled to generalize on the
small 82-month aggregated series, a useful learning point on the data volume SARIMA needs
to perform well. Week 3 — Customer Intelligence System (Clustering + Ensemb

## 11. Observations

A few things worth noting from putting this together:

- Chunk size makes a real difference in retrieval quality. Too small
  (150 chars) and chunks start losing enough context to be useful on
  their own; too large (600 chars) and a chunk can straddle multiple
  topics, which drags down how targeted the retrieval match is. 300
  characters was a reasonable middle ground for this document, but
  that's specific to how densely this text is written and would need
  re-checking on a longer, more varied document.
- Pure vector search occasionally pulled a chunk that was topically
  close but didn't actually contain the specific keyword the question
  was about. The hybrid scoring in 10.2 caught some of those cases,
  though it's a fairly blunt keyword-overlap heuristic rather than
  anything like BM25.
- The cross-encoder re-ranker in 10.3 changed the ordering of results
  compared to plain vector similarity more often than expected, which
  makes sense since it's actually reading the question and the chunk
  together rather than comparing two independently-computed embeddings.
  The tradeoff is speed - it has to score every candidate individually,
  so it only makes sense as a second pass over a small pool, not as the
  primary retrieval step over the whole document.
- `flan-t5-base` keeps answers grounded in the retrieved context most of
  the time, but it's a small model and its answers can be terse or
  slightly off when the retrieved chunks don't fully cover the question.
  A larger model would very likely improve fluency without changing
  anything else in the pipeline, since retrieval and generation are
  decoupled by design.
- The main failure mode wasn't the retrieval step, it was questions that
  don't map cleanly onto any single chunk and need information stitched
  together from two different chunks. That's a reasonable next thing to
  test with a longer, multi-page real PDF instead of the short demo
  document used as a fallback here.

## 12. Conclusion

This notebook builds a complete RAG pipeline: a document (uploaded PDF,
uploaded text file, or a Hugging Face dataset) is ingested and chunked,
chunks are embedded and stored in ChromaDB, questions are embedded with
the same model, relevant chunks are retrieved, and a language model
generates an answer grounded in that retrieved context rather than
relying on its own internal knowledge. The system metrics report
documents exactly what was used at each stage, and the optimization
experiments cover all three directions suggested in the assignment -
chunk size, hybrid keyword+vector search, and a real cross-encoder
re-ranking model - rather than just the baseline pipeline on its own.